<a href="https://colab.research.google.com/github/KosTeS1/test-external/blob/new-branch/Homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Задание 1. Базовое изучение**
---
Решение:
1. Выгружаем название столбцов из файла horse_data.names по ссылке
2. Выгрузка из файла horse_data.csv
3. Для анализа берём 4 числовых столбца и 4 категорийных

In [3]:
import pandas as pd
import numpy as np

# 1. Загрузка названий столбцов с по ссылке
url = "https://raw.githubusercontent.com/obulygin/pyda_homeworks/master/statistics_basics/horse_data.names"
column_names = [
    'surgery', 'age', 'hospital_number', 'rectal_temp', 'pulse', 'respiratory_rate',
    'temp_of_extremities', 'peripheral_pulse', 'mucous_membranes', 'capillary_refill_time',
    'pain', 'peristalsis', 'abdominal_distension', 'nasogastric_tube', 'nasogastric_reflux',
    'nasogastric_reflux_ph', 'rectal_exam_feces', 'abdomen', 'packed_cell_volume',
    'total_protein', 'abdomo_appearance', 'abdomo_protein', 'outcome', 'surgical_lesion',
    'lesion_1', 'lesion_2', 'lesion_3', 'cp_data'
]

# 2. Загрузка данных
df = pd.read_csv('horse_data.csv', na_values='?', header=None)
df.columns = column_names

# 3. Выбор 8 столбцов для анализа
selected_cols = [
    'rectal_temp', 'pulse', 'packed_cell_volume', 'total_protein',  # числовые
    'surgery', 'pain', 'abdominal_distension', 'surgical_lesion'     # категориальные
]
data = df[selected_cols]

print("АНАЛИЗ ДАННЫХ О ЛОШАДЯХ")
print("=" * 40)

# 4. Анализ числовых столбцов
print("\nЧИСЛОВЫЕ СТОЛБЦЫ:")
numeric_data = data[['rectal_temp', 'pulse', 'packed_cell_volume', 'total_protein']]
print(numeric_data.describe())

# 5. Анализ категориальных столбцов
print("\nКАТЕГОРИАЛЬНЫЕ СТОЛБЦЫ:")
for col in ['surgery', 'pain', 'abdominal_distension', 'surgical_lesion']:
    mode_val = data[col].mode()[0]
    print(f"{col}: мода = {mode_val}")

# 6. Пропуски
print(f"\nПРОПУСКИ: {data.isnull().sum().sum()} значений")

АНАЛИЗ ДАННЫХ О ЛОШАДЯХ

ЧИСЛОВЫЕ СТОЛБЦЫ:
       rectal_temp       pulse  packed_cell_volume  total_protein
count   240.000000  276.000000          271.000000     267.000000
mean     38.167917   71.913043           46.295203      24.456929
std       0.732289   28.630557           10.419335      27.475009
min      35.400000   30.000000           23.000000       3.300000
25%      37.800000   48.000000           38.000000       6.500000
50%      38.200000   64.000000           45.000000       7.500000
75%      38.500000   88.000000           52.000000      57.000000
max      40.800000  184.000000           75.000000      89.000000

КАТЕГОРИАЛЬНЫЕ СТОЛБЦЫ:
surgery: мода = 1.0
pain: мода = 3.0
abdominal_distension: мода = 1.0
surgical_lesion: мода = 1

ПРОПУСКИ: 258 значений


**Задание 2. Работа с выбросами**

---
1. Проверяем выбросы по числовым значениям через метод IQR
2. Делаем предварительные выводы

In [15]:
print("\n" + "=" * 50)
print("ЗАДАНИЕ 2: АНАЛИЗ ВЫБРОСОВ МЕТОДОМ IQR")
print("=" * 50)

def find_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return outliers, lower_bound, upper_bound

# Анализ выбросов
numeric_cols = ['rectal_temp', 'pulse', 'packed_cell_volume', 'total_protein']

for col in numeric_cols:
    clean_data = data[col].dropna()
    outliers, lower_bound, upper_bound = find_outliers_iqr(clean_data)

    print(f"\n{col.upper()}:")
    print(f"  Границы: [{lower_bound:.1f}, {upper_bound:.1f}]")
    print(f"  Выбросов: {len(outliers)}")

print("\n" + "=" * 50)
print("""
1. PULSE - больше всего выбросов (высокий пульс)
   Возможная причина: шок, боль, стресс у лошадей

2. RECTAL_TEMP - несколько выбросов
   Возможная причина: инфекция

3. PACKED_CELL_VOLUME - несколько выбросов
   Возможная причина: обезвоживание

4. TOTAL_PROTEIN - нет выбросов
   Значения в норме""")

print("ВЫВОД: Лошади испытывают стресс, а некоторые болеют")


ЗАДАНИЕ 2: АНАЛИЗ ВЫБРОСОВ МЕТОДОМ IQR

RECTAL_TEMP:
  Границы: [36.7, 39.6]
  Выбросов: 14

PULSE:
  Границы: [-12.0, 148.0]
  Выбросов: 5

PACKED_CELL_VOLUME:
  Границы: [17.0, 73.0]
  Выбросов: 3

TOTAL_PROTEIN:
  Границы: [-69.2, 132.8]
  Выбросов: 0


1. PULSE - больше всего выбросов (высокий пульс)
   Возможная причина: шок, боль, стресс у лошадей

2. RECTAL_TEMP - несколько выбросов
   Возможная причина: инфекция

3. PACKED_CELL_VOLUME - несколько выбросов
   Возможная причина: обезвоживание

4. TOTAL_PROTEIN - нет выбросов
   Значения в норме
ВЫВОД: Лошади испытывают стресс, а некоторые болеют


**Задание 3. Работа с пропусками**

---

1.   Посчитаем пропуски в каждом столбце
2.   Выбираем методы заполнения
3.   Заполняем пропуски

In [22]:
print("\n" + "=" * 50)
print("ЗАДАНИЕ 3: РАБОТА С ПРОПУСКАМИ")
print("=" * 50)

# 1. Расчет пропусков
print("КОЛИЧЕСТВО ПРОПУСКОВ:")
missing_counts = data.isnull().sum()
print(missing_counts)
print(f"\nВсего пропусков: {missing_counts.sum()}")

# 2. Решение по методам обработки
print("\nОБОСНОВАНИЕ РЕШЕНИЯ:")
print("• Числовые: заполняем медианой. Медиана для числовых - потому что в pulse есть выбросы, которые исказят среднее")
print("• Категориальные: заполняем модой. Мода для категориальных - потому что это наиболее частые категории")

# 3. Создание датафрейма без пропусков (без inplace)
data_no_missing = data.copy()

# Заполнение числовых столбцов
numeric_cols = ['rectal_temp', 'pulse', 'packed_cell_volume', 'total_protein']
for col in numeric_cols:
    median_val = data[col].median()
    data_no_missing[col] = data_no_missing[col].fillna(median_val)

# Заполнение категориальных столбцов
categorical_cols = ['surgery', 'pain', 'abdominal_distension', 'surgical_lesion']
for col in categorical_cols:
    mode_val = data[col].mode()[0]
    data_no_missing[col] = data_no_missing[col].fillna(mode_val)

print(f"\nПропусков после обработки: {data_no_missing.isnull().sum().sum()}")


ЗАДАНИЕ 3: РАБОТА С ПРОПУСКАМИ
КОЛИЧЕСТВО ПРОПУСКОВ:
rectal_temp             60
pulse                   24
packed_cell_volume      29
total_protein           33
surgery                  1
pain                    55
abdominal_distension    56
surgical_lesion          0
dtype: int64

Всего пропусков: 258

ОБОСНОВАНИЕ РЕШЕНИЯ:
• Числовые: заполняем медианой. Медиана для числовых - потому что в pulse есть выбросы, которые исказят среднее
• Категориальные: заполняем модой. Мода для категориальных - потому что это наиболее частые категории

Пропусков после обработки: 0
